импортируем все нужное и прочитаем данные. посмотрим, сколько строк в каждом файле и как часто локация запроса совпадает с локацией выбранного объявления. это поможет понять, стоит ли учитывать такое совпадение при поиске

In [1]:
import gc
import re
import time
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from nltk.stem.snowball import RussianStemmer
from tqdm import tqdm

started = time.time()
data_dir = Path('data')
search_cols = [
    'search_query', 'search_location_id', 'search_is_delivery_search',
    'search_infm_params_text', 'search_category',
]
item_cols = [
    'item_id', 'item_title_raw', 'item_description_raw', 'item_infm_params_text',
    'item_location_id', 'item_latitude', 'item_longitude',
]

train = pd.read_parquet(data_dir / 'train.parquet', columns=search_cols + item_cols)
queries = pd.read_parquet(data_dir / 'benchmark_queries.parquet')
benchmark_items = pd.read_parquet(data_dir / 'benchmark_items.parquet', columns=item_cols)
print('строк train:', len(train))
print('запросов benchmark:', len(queries))
print('объявлений benchmark:', len(benchmark_items))
print('совпадение локаций в train:', f'{(train.search_location_id == train.item_location_id).mean():.1%}')

строк train: 497673
запросов benchmark: 2452
объявлений benchmark: 189212
совпадение локаций в train: 83.1%


сначала подготовим текст: переведем его в нижний регистр, уберем знаки препинания и лишние пробелы. для поиска по словам будем обрезать окончания с помощью стемминга. так разные формы одного слова чаще будут совпадать при поиске

уберем повторяющиеся пары запроса и объявления. затем соберем объявления из train и benchmark в одну таблицу, чтобы искать среди них при проверке решения. каждый item_id оставим один раз. если объявление есть в обоих файлах, возьмем его данные из benchmark

еще нам понадобится примерный центр каждой локации. возьмем медиану широты и долготы ее объявлений. дальше от этой точки будем считать расстояние до объявления

In [2]:
def clean(text):
    text = str(text).lower().replace('\u0451', 'е').replace('\\n', ' ')
    return ' '.join(re.findall(r'[а-яa-z0-9]+', text))

stemmer = RussianStemmer()

@lru_cache(maxsize=150000)
def stem(word):
    return stemmer.stem(word)


def tokenize(text):
    return [stem(word) for word in clean(text).split() if len(word) > 1]


train['query_text'] = train.search_query.map(clean)
interactions = train[search_cols + ['query_text', 'item_id']].drop_duplicates().copy()
all_items = pd.concat([train[item_cols], benchmark_items], ignore_index=True)
all_items = all_items.drop_duplicates('item_id', keep='last').reset_index(drop=True)
del train
gc.collect()


all_items[['item_latitude', 'item_longitude']] = all_items[['item_latitude', 'item_longitude']].astype(float)
centers = all_items.groupby('item_location_id')[['item_latitude', 'item_longitude']].median()

known_queries = set(interactions.query_text)
print('новых текстов в benchmark:', f'{(~queries.search_query.map(clean).isin(known_queries)).mean():.1%}')
print('объявлений в корпусе для проверки:', len(all_items))

новых текстов в benchmark: 62.5%
объявлений в корпусе для проверки: 515895


проверим, все ли локации из запросов встречаются среди объявлений. если такой локации нет, мы не сможем ни найти объявления с тем же item_location_id, ни определить ее центр для расчета расстояния

In [3]:
missing_location = ~queries.search_location_id.isin(centers.index)
print('запросов без прямого соответствия локации:', int(missing_location.sum()))
print('доля таких запросов:', f'{missing_location.mean():.1%}')


item_geo = all_items[['item_id', 'item_location_id', 'item_latitude', 'item_longitude']].copy()

запросов без прямого соответствия локации: 425
доля таких запросов: 17.3%


таких запросов оказалось 425, или 17,3%. попробуем помочь им с помощью train: посмотрим, в каких локациях пользователи выбирали объявления, когда искали в этой локации

перед этим разделим данные. на 1800 запросах будем сравнивать настройки, еще 1200 оставим для проверки уже выбранного решения. делим по очищенному тексту запроса. если текст попал в проверку, убираем из обучения все строки с этим текстом, даже если у них другие фильтры или локация. так мы не будем подсказывать модели ответ через историю того же запроса. тексты, которые использовались в прошлых экспериментах, тоже не возвращаем в обучение

один текст может встречаться с разными фильтрами и локациями. для проверки случайно оставим один такой вариант на каждый текст. правильными ответами для него будут все объявления, выбранные именно при этих условиях

In [4]:
rng = np.random.default_rng(42)
texts = interactions.query_text.unique()
first_texts = rng.choice(texts, 2400, replace=False)
remaining = texts[~pd.Index(texts).isin(first_texts)]
previous_test_texts = rng.choice(remaining, 1200, replace=False)
used_texts = set(first_texts) | set(previous_test_texts)

used_rows = interactions[interactions.query_text.isin(used_texts)]
used_queries = used_rows.groupby(search_cols, sort=False, dropna=False).agg(
    relevant=('item_id', lambda values: set(values)),
    query_text=('query_text', 'first'),
).reset_index()
used_queries = used_queries.sample(frac=1, random_state=42).drop_duplicates('query_text')

first_part = used_queries[used_queries.query_text.isin(first_texts)].sample(n=600, random_state=42)
second_part = used_queries[used_queries.query_text.isin(previous_test_texts)].sample(frac=1, random_state=42)
validation = pd.concat([second_part, first_part], ignore_index=True)

remaining = texts[~pd.Index(texts).isin(used_texts)]
test_texts = np.random.default_rng(314).choice(remaining, 1200, replace=False)
test_rows = interactions[interactions.query_text.isin(test_texts)]
test = test_rows.groupby(search_cols, sort=False, dropna=False).agg(
    relevant=('item_id', lambda values: set(values)),
    query_text=('query_text', 'first'),
).reset_index()
test = test.sample(frac=1, random_state=42).drop_duplicates('query_text')
test = test.sample(frac=1, random_state=43).reset_index(drop=True)

held_texts = used_texts | set(test_texts)
fit = interactions[~interactions.query_text.isin(held_texts)].copy()
assert not set(fit.query_text) & held_texts
assert not set(validation.query_text) & set(test.query_text)
print('обучающих пар:', len(fit))
print('для сравнения вариантов:', len(validation))
print('для итоговой проверки:', len(test))

обучающих пар: 441978
для сравнения вариантов: 1800
для итоговой проверки: 1200


для каждого объявления соберем текст из заголовка, параметров, описания и запросов, по которым его выбирали в обучающей части. история запросов полезна тем, что пользователь может называть услугу иначе, чем автор объявления

заголовок повторим три раза, а запросы из истории два раза. так слова из этих частей будут сильнее влиять на оценку. возьмем первые 400 символов параметров, первые 1400 символов описания и до восьми разных запросов из истории

по этим текстам построим индекс bm25. он позволит оценить совпадение слов запроса со словами объявления. редкие слова получают больший вес, а длина текста учитывается, чтобы длинные описания не выигрывали только за счет объема. в формуле используем k1=1.5 и b=0.75

In [5]:
def build_word_index(items, history, description_length=1400, min_df=2, max_features=200000):
    known_queries = history.groupby('item_id').query_text.agg(
        lambda values: ' '.join(pd.unique(values)[:8])
    )
    documents = (items.item_title_raw.fillna('') + ' ') * 3
    documents += items.item_infm_params_text.fillna('').str[:400] + ' '
    documents += items.item_description_raw.fillna('').str[:description_length] + ' '
    documents += (items.item_id.map(known_queries).fillna('') + ' ') * 2

    vectorizer = CountVectorizer(
        tokenizer=tokenize, token_pattern=None, lowercase=False,
        min_df=min_df, max_df=0.9, max_features=max_features, dtype=np.float32,
    )
    counts = vectorizer.fit_transform(tqdm(documents, desc='слова', mininterval=20))
    lengths = np.asarray(counts.sum(axis=1)).ravel()
    document_frequency = np.bincount(counts.indices, minlength=counts.shape[1])
    idf = np.log1p((len(items) - document_frequency + 0.5) / (document_frequency + 0.5))

    k1 = 1.5
    b = 0.75
    length_norm = k1 * (1 - b + b * lengths / lengths.mean())
    entries = counts.tocoo()
    values = entries.data.copy()
    values *= (k1 + 1) / (entries.data + length_norm[entries.row])
    values *= idf[entries.col]
    word_index = sparse.csr_matrix(
        (values, (entries.col, entries.row)), shape=(counts.shape[1], len(items))
    )
    return vectorizer, word_index

word_vectorizer, word_index = build_word_index(all_items, fit)

слова:   0%|          | 0/515895 [00:00<?, ?it/s]

слова:  19%|█▊        | 95950/515895 [00:20<01:27, 4797.47it/s]

слова:  19%|█▊        | 95950/515895 [00:30<01:27, 4797.47it/s]

слова:  38%|███▊      | 196027/515895 [00:40<01:05, 4918.85it/s]

слова:  38%|███▊      | 196027/515895 [00:50<01:05, 4918.85it/s]

...

слова:  57%|█████▋    | 296358/515895 [01:10<00:44, 4963.45it/s]

слова:  76%|███████▋  | 393644/515895 [01:20<00:24, 4924.30it/s]

слова:  76%|███████▋  | 393644/515895 [01:30<00:24, 4924.30it/s]

слова:  95%|█████████▍| 489339/515895 [01:40<00:05, 4873.93it/s]

слова: 100%|██████████| 515895/515895 [01:45<00:00, 4890.89it/s]

добавим еще один способ сравнивать запрос с заголовком. будем смотреть на совпадения кусочков слов длиной от 3 до 5 символов и считать их оценки через tf idf

если в слове есть опечатка или другое окончание, часть таких кусочков все равно совпадет. это может помочь там, где поиск по словам пропускает нужное объявление

In [6]:
char_vectorizer = TfidfVectorizer(
    preprocessor=clean, analyzer='char_wb', ngram_range=(3, 5),
    min_df=3, max_features=180000, sublinear_tf=True, dtype=np.float32,
)
char_matrix = char_vectorizer.fit_transform(all_items.item_title_raw.fillna(''))
char_index = char_matrix.T.tocsr()
del char_matrix

baseline_index = {
    'words': word_vectorizer, 'word_index': word_index,
    'chars': char_vectorizer, 'char_index': char_index,
}

теперь посчитаем связь между локацией поиска и локацией выбранного объявления. для каждой локации поиска возьмем все выборы из обучающей части и посмотрим, какая доля пришлась на каждую локацию объявлений

например, если из 100 выборов 30 пришлись на одну локацию, ее доля будет 0.3. дальше будем использовать эти доли, чтобы повышать оценки объявлений из подходящих локаций. одинаковые идентификаторы локаций для этого не нужны. строки с отложенными запросами в расчет не берем

In [7]:
def get_location_probabilities(history):
    pairs = history[['search_location_id', 'item_id']].merge(
        item_geo[['item_id', 'item_location_id']], on='item_id', how='left'
    )
    counts = pairs.groupby(['search_location_id', 'item_location_id']).size()
    totals = counts.groupby(level=0).transform('sum')
    return counts / totals

location_probabilities = get_location_probabilities(fit)

расстояние тоже учтем: чем ближе объявление к центру локации поиска, тем больше прибавка к его оценке. с расстоянием эта прибавка плавно уменьшается. при расстоянии 40 км она составляет примерно 0.37 от значения в самом центре

40 здесь задает, насколько быстро уменьшается прибавка. объявления дальше 40 км мы не отбрасываем. если центр локации неизвестен, прибавка за расстояние будет нулевой

In [8]:
def distance_bonus(location, latitude, longitude):
    if location not in centers.index:
        return np.zeros(len(latitude), dtype=np.float32)

    query_latitude, query_longitude = np.radians(centers.loc[location].to_numpy())
    a = np.sin((latitude - query_latitude) / 2) ** 2
    a += np.cos(query_latitude) * np.cos(latitude) * np.sin((longitude - query_longitude) / 2) ** 2
    distance = 6371 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    return np.nan_to_num(np.exp(-distance / 40), nan=0).astype(np.float32)

соберем все оценки вместе. сначала для каждого запроса поделим оценки bm25 на самую большую из них. то же самое сделаем с оценками по кусочкам слов и по тексту фильтров. после этого каждая оценка будет от 0 до 1, и их будет проще смешать

в первом варианте дадим поиску по словам вес 0.7, а поиску по кусочкам слов 0.3. совпадения с текстом фильтров добавим с небольшим весом 0.05

затем увеличим полученную оценку с учетом локации. к множителю добавим 7, если локации совпали. если они разные, добавим до 3 в зависимости от расстояния. отдельно добавим долю выборов из истории с весом, который подберем дальше. из доли возьмем квадратный корень: так разница между частыми и редкими локациями станет меньше

обработаем запросы по одному и для каждого возьмем 50 объявлений с самой большой итоговой оценкой

In [9]:
def predict(query_frame, items, index, probabilities, variants):
    word_queries = index['words'].transform(query_frame.search_query).tocsr()
    filter_queries = index['words'].transform(query_frame.search_infm_params_text).tocsr()
    char_queries = index['chars'].transform(query_frame.search_query).tocsr()
    word_queries.data[:] = 1
    filter_queries.data[:] = 1

    item_ids = items.item_id.to_numpy()
    locations = items.item_location_id.to_numpy()
    city_ids, city_codes = np.unique(locations, return_inverse=True)
    latitude = np.radians(items.item_latitude.to_numpy(dtype=np.float32))
    longitude = np.radians(items.item_longitude.to_numpy(dtype=np.float32))
    known_locations = set(probabilities.index.get_level_values(0))
    predictions = [[] for _ in range(len(variants))]

    for i, location in enumerate(tqdm(query_frame.search_location_id, desc='поиск', mininterval=20)):
        word = (word_queries[i] @ index['word_index']).toarray().ravel()
        char = (char_queries[i] @ index['char_index']).toarray().ravel()
        filters = (filter_queries[i] @ index['word_index']).toarray().ravel()
        word /= max(word.max(), 1e-9)
        char /= max(char.max(), 1e-9)
        filters /= max(filters.max(), 1e-9)

        same_location = locations == location
        nearby = distance_bonus(location, latitude, longitude) * (~same_location)
        history_score = np.zeros(len(items), dtype=np.float32)
        if location in known_locations:
            city_probabilities = probabilities.loc[location].reindex(city_ids, fill_value=0)
            history_score = np.sqrt(city_probabilities.to_numpy(dtype=np.float32)[city_codes])

        for j, variant in enumerate(variants.itertuples(index=False)):
            text_score = variant.word_weight * word + (1 - variant.word_weight) * char + 0.05 * filters
            score = text_score * (1 + 7 * same_location + 3 * nearby + variant.history_weight * history_score)
            top = np.argpartition(score, -50)[-50:]
            top = top[np.argsort(-score[top], kind='stable')]
            predictions[j].append(item_ids[top].tolist())
    return predictions


def recall_at_50(predictions, relevant):
    values = [len(set(found) & correct) / len(correct) for found, correct in zip(predictions, relevant)]
    return np.mean(values)

проверим, насколько сильно стоит учитывать историю локаций. сравним веса 0, 1, 3 и 7 на тех 1800 запросах, которые оставили для подбора настроек. при весе 0 история локаций не влияет на результат, поэтому это будет вариант для сравнения. дальше оставим вес с самым большим recall@50

In [10]:
variants = pd.DataFrame({
    'word_weight': [0.7, 0.7, 0.7, 0.7],
    'history_weight': [0, 1, 3, 7],
})
validation_predictions = predict(validation, all_items, baseline_index, location_probabilities, variants)
validation_scores = [recall_at_50(found, validation.relevant) for found in validation_predictions]
comparison = variants.copy()
comparison['recall@50'] = validation_scores
display(comparison)

best_row = int(np.argmax(validation_scores))
best_variant = variants.iloc[[best_row]].reset_index(drop=True)
best_index = baseline_index
best_description_length = 1400
best_min_df = 2
best_max_features = 200000
best_validation_score = validation_scores[best_row]
print('лучший recall@50 на сравнении:', round(best_validation_score, 4))

поиск:   0%|          | 0/1800 [00:00<?, ?it/s]

поиск:  22%|██▏       | 387/1800 [00:20<01:13, 19.34it/s]

поиск:  22%|██▏       | 387/1800 [00:30<01:13, 19.34it/s]

поиск:  47%|████▋     | 842/1800 [00:40<00:44, 21.33it/s]

поиск:  47%|████▋     | 842/1800 [00:50<00:44, 21.33it/s]

...

поиск:  89%|████████▉ | 1610/1800 [01:20<00:09, 19.75it/s]

поиск:  89%|████████▉ | 1610/1800 [01:30<00:09, 19.75it/s]

поиск: 100%|██████████| 1800/1800 [01:30<00:00, 19.85it/s]

,word_weight,history_weight,recall@50
0,0.7,0,0.742315
1,0.7,1,0.778426
2,0.7,3,0.781343
3,0.7,7,0.776250


лучший recall@50 на сравнении: 0.7813


теперь попробуем брать больше текста из описания: 4500 символов вместо 1400. еще сохраним слова, которые встретились всего в одном объявлении. среди них могут быть редкие названия услуг или важные детали

увеличим предел словаря до 350 тысяч слов. выбранный вес истории локаций оставим, а для поиска по словам проверим веса 0.7, 0.9 и 0.5. вес поиска по кусочкам слов каждый раз будет дополнять его до 1. если новый вариант даст результат лучше, сохраним его настройки

In [11]:
long_words, long_word_index = build_word_index(
    all_items, fit, description_length=4500, min_df=1, max_features=350000
)
long_index = {
    'words': long_words, 'word_index': long_word_index,
    'chars': char_vectorizer, 'char_index': char_index,
}
long_variants = pd.DataFrame({
    'word_weight': [0.7, 0.9, 0.5],
    'history_weight': [float(best_variant.history_weight.iloc[0])] * 3,
})
long_predictions = predict(validation, all_items, long_index, location_probabilities, long_variants)
long_scores = [recall_at_50(found, validation.relevant) for found in long_predictions]
long_comparison = long_variants.copy()
long_comparison['recall@50'] = long_scores
display(long_comparison)

long_best_row = int(np.argmax(long_scores))
if long_scores[long_best_row] > best_validation_score:
    best_index = long_index
    best_variant = long_variants.iloc[[long_best_row]].reset_index(drop=True)
    best_description_length = 4500
    best_min_df = 1
    best_max_features = 350000
    best_validation_score = long_scores[long_best_row]

print('длина описания:', best_description_length)
print('выбранные веса:')
display(best_variant)

слова:   0%|          | 0/515895 [00:00<?, ?it/s]

слова:  13%|█▎        | 69204/515895 [00:20<02:09, 3460.19it/s]

слова:  13%|█▎        | 69204/515895 [00:35<02:09, 3460.19it/s]

слова:  27%|██▋       | 137669/515895 [00:40<01:49, 3438.44it/s]

слова:  27%|██▋       | 137669/515895 [00:55<01:49, 3438.44it/s]

...

поиск:  94%|█████████▎| 1684/1800 [01:20<00:05, 20.87it/s]

поиск: 100%|██████████| 1800/1800 [01:25<00:00, 20.98it/s]

,word_weight,history_weight,recall@50
0,0.7,3.0,0.799028
1,0.9,3.0,0.794861
2,0.5,3.0,0.783287


длина описания: 4500
выбранные веса:


,word_weight,history_weight
0,0.7,3.0


настройки выбрали. теперь сравним исходный и выбранный варианты на 1200 запросах, которые до этого не использовали для сравнения. оба варианта получат одну и ту же обучающую историю и будут искать среди одних и тех же объявлений

еще отдельно посмотрим на запросы, у которых ни одно правильное объявление не встречалось в обучающей истории. у таких объявлений нет прошлых запросов, которые можно добавить к их тексту. эта проверка покажет, как решение справляется с ними

по результату этой проверки настройки уже не меняем

In [12]:
baseline_variant = pd.DataFrame({'word_weight': [0.7], 'history_weight': [0]})
baseline_test_predictions = predict(test, all_items, baseline_index, location_probabilities, baseline_variant)[0]
best_test_predictions = predict(test, all_items, best_index, location_probabilities, best_variant)[0]
baseline_recall = recall_at_50(baseline_test_predictions, test.relevant)
best_recall = recall_at_50(best_test_predictions, test.relevant)
display(pd.DataFrame({
    'вариант': ['исходный', 'улучшенный'],
    'recall@50': [baseline_recall, best_recall],
}))
print('прирост:', round(best_recall - baseline_recall, 4))

known_items = set(fit.item_id)
cold = test.relevant.map(lambda correct: correct.isdisjoint(known_items)).to_numpy()
cold_predictions = [found for found, is_cold in zip(best_test_predictions, cold) if is_cold]
print('запросов только с новыми объявлениями:', int(cold.sum()))
print('recall@50 на них:', round(recall_at_50(cold_predictions, test.loc[cold, 'relevant']), 4))

поиск:   0%|          | 0/1200 [00:00<?, ?it/s]

поиск:  52%|█████▏    | 627/1200 [00:20<00:18, 31.31it/s]

поиск:  52%|█████▏    | 627/1200 [00:35<00:18, 31.31it/s]

поиск: 100%|██████████| 1200/1200 [00:38<00:00, 31.42it/s]

поиск:   0%|          | 0/1200 [00:00<?, ?it/s]

поиск:  58%|█████▊    | 701/1200 [00:20<00:14, 35.04it/s]

поиск: 100%|██████████| 1200/1200 [00:34<00:00, 34.32it/s]

,вариант,recall@50
0,исходный,0.744861
1,улучшенный,0.798750


прирост: 0.0539
запросов только с новыми объявлениями: 723
recall@50 на них: 0.7842


теперь подготовим итоговый ответ. для поиска оставим только объявления из benchmark_items. запросы из истории и доли выборов по локациям пересчитаем на всем train, потому что локальная проверка уже закончена

применим выбранные настройки к benchmark_queries и сохраним найденные item_id в answer.csv

In [13]:
del baseline_index, best_index, long_index, word_index, long_word_index, char_index, all_items

gc.collect()
stem.cache_clear()

final_words, final_word_index = build_word_index(
    benchmark_items, interactions, description_length=best_description_length,
    min_df=best_min_df, max_features=best_max_features,
)
final_chars = TfidfVectorizer(
    preprocessor=clean, analyzer='char_wb', ngram_range=(3, 5),
    min_df=3, max_features=180000, sublinear_tf=True, dtype=np.float32,
)
final_char_index = final_chars.fit_transform(benchmark_items.item_title_raw.fillna('')).T.tocsr()
final_index = {
    'words': final_words, 'word_index': final_word_index,
    'chars': final_chars, 'char_index': final_char_index,
}
final_location_probabilities = get_location_probabilities(interactions)
predictions = predict(queries, benchmark_items, final_index, final_location_probabilities, best_variant)[0]
answer = pd.DataFrame({
    'query_id': queries.query_id,
    'answer': [' '.join(item_ids) for item_ids in predictions],
})
answer.to_csv('answer.csv', index=False, encoding='utf-8')

слова:   0%|          | 0/189212 [00:00<?, ?it/s]

слова:  34%|███▍      | 65254/189212 [00:20<00:37, 3262.63it/s]

слова:  34%|███▍      | 65254/189212 [00:39<00:37, 3262.63it/s]

слова:  69%|██████▉   | 131372/189212 [00:40<00:17, 3288.06it/s]

слова:  69%|██████▉   | 131372/189212 [00:50<00:17, 3288.06it/s]

слова: 100%|██████████| 189212/189212 [00:56<00:00, 3345.08it/s]

поиск:   0%|          | 0/2452 [00:00<?, ?it/s]

поиск:  85%|████████▍ | 2082/2452 [00:20<00:03, 104.08it/s]

поиск: 100%|██████████| 2452/2452 [00:23<00:00, 105.57it/s]

прочитаем сохраненный answer.csv и проверим, что в нем ровно нужные query_id и каждый встречается один раз. в каждом ответе должно быть не больше 50 разных item_id, и все они должны быть в benchmark_items

еще проверим названия колонок и запись самих идентификаторов. при чтении явно оставим их строками, чтобы не потерять ведущие нули

In [14]:
saved = pd.read_csv('answer.csv', dtype=str, keep_default_na=False, encoding='utf-8')
assert saved.columns.tolist() == ['query_id', 'answer']
assert len(saved) == len(queries)
assert saved.query_id.is_unique
assert saved.query_id.str.len().eq(16).all()
assert set(saved.query_id) == set(queries.query_id)

allowed_ids = set(benchmark_items.item_id)
for value in saved.answer:
    item_ids = value.split(' ')
    assert 1 <= len(item_ids) <= 50
    assert len(item_ids) == len(set(item_ids))
    assert all(re.fullmatch(r'[0-9a-f]{16}', item_id) for item_id in item_ids)
    assert set(item_ids) <= allowed_ids

print('формат проверен:', len(saved), 'строки, по 50 уникальных item_id.')
print('файл:', Path('answer.csv').resolve())
print('время выполнения:', round((time.time() - started) / 60, 1), 'мин.')
saved.head(3)

формат проверен: 2452 строки, по 50 уникальных item_id.
файл: /Applications/programming/vscode_projects/candidate-generation-for-a-service-category/answer.csv
время выполнения: 11.3 мин.


,query_id,answer
0,70DfDUpwjxB4lzFd,b92ee8f432cec2d1 d722bcda1a555091 255fbeaf526a...
1,JTrdTaZJvSiLPkXj,422d3ffdd5bbf626 367af128a9ea2a48 cbeccbecb1fb...
2,LZCZNoVG4AFUkVRJ,3f89b8062dc85f1c dab52187b4500d9b d8fce513e4f0...


для повторного запуска: python 3.11 и следующие версии зависимостей
```bash
pip install numpy==2.4.6 pandas==3.0.5 scipy==1.17.1 scikit-learn==1.9.1 nltk==3.10.3 pyarrow==25.0.1 tqdm==4.70.1
```
использованные готовые методы: bm25, символьный tf idf из [scikit learn](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction), русский snowball из [nltk](https://www.nltk.org/api/nltk.stem.snowball.html)